# 🚀 AI LVQ Std BOT — Monthly Specialist Fine-Tuning (`kingescaped/lvq-std-BOT`)

Notebook huấn luyện tinh chỉnh (**Fine-tune**) tự động hàng tháng trên **Kaggle GPU T4** theo cơ chế **Champion vs. Challenger (cùng đề thi 30 ngày mới nhất)** cho 2 nhánh chuyên biệt trên repo Private Hugging Face **`kingescaped/lvq-std-BOT`**:
- **`btc-m15`**: Kéo tự động **1 năm (365 ngày)** nến M15 đã đóng của Bitcoin từ **Binance Public API** (`data-api.binance.vision`), tách **30 ngày mới nhất** làm đề thi chung.
- **`xau-h1`**: Kéo tự động **2 năm (`730d`)** nến H1 đã đóng của Vàng từ **Yahoo Finance (`GC=F`)**, tách **30 ngày mới nhất** làm đề thi chung.

### Thiết lập Kaggle Secrets 1 lần duy nhất:
Mở Notebook này ở chế độ **Edit → Add-ons → Secrets**, thêm và tích chọn gắn vào Notebook:
- `HF_TOKEN` *(bắt buộc)*: Token quyền **Write** tại `https://huggingface.co/settings/tokens`.
- `TELEGRAM_BOT_TOKEN` & `TELEGRAM_ADMIN_ID` *(cần cả 2 để nhận tin nhắn riêng định kỳ hàng tháng)*: Bấm `/start` với Bot trên Telegram từ tài khoản `@kingescaped`, lấy mã số User ID (qua `@userinfobot`) điền vào `TELEGRAM_ADMIN_ID`.

In [ ]:
!pip install -q huggingface_hub safetensors einops tqdm yfinance "pandas>=2.0.0,<3.0.0" "six>=1.16.0" requests

import os
import subprocess

WORK_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = os.path.join(WORK_ROOT, 'xauusd_signal_bot')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/kingescaped/xauusd_signal_bot.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)

In [ ]:
import os
import torch


def _read_secret(secret_name):
    env_val = (os.getenv(secret_name) or '').strip()
    if env_val:
        return env_val
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(secret_name) or '').strip()
    except Exception:
        pass
    try:
        from google.colab import userdata
        return (userdata.get(secret_name) or '').strip()
    except Exception:
        return ''


HF_TOKEN = _read_secret('HF_TOKEN')
TELEGRAM_BOT_TOKEN = _read_secret('TELEGRAM_BOT_TOKEN')
TELEGRAM_ADMIN_ID = _read_secret('TELEGRAM_ADMIN_ID')
TELEGRAM_GROUP_ID = _read_secret('TELEGRAM_GROUP_ID')
TARGET_REPO = 'kingescaped/lvq-std-BOT'

gpu_label = f'GPU ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else 'CPU'
print('Thiết bị huấn luyện:', gpu_label)

READY_TO_TRAIN = bool(HF_TOKEN)
if not READY_TO_TRAIN:
    print(
        '[CẦN CẤU HÌNH 1 LẦN] Chưa tìm thấy HF_TOKEN trong Kaggle Secrets.\n'
        '-> Mở Notebook này trên Kaggle -> Edit -> Add-ons -> Secrets -> thêm HF_TOKEN (quyền Write) và tích chọn gắn vào Notebook!'
    )
    if TELEGRAM_BOT_TOKEN:
        from telegram_notifier import send_telegram_direct_message
        send_telegram_direct_message(
            '⚠️ <b>[AI LVQ Std BOT]</b> Chưa tìm thấy <code>HF_TOKEN</code> trong Kaggle Secrets, bỏ qua lượt fine-tune tháng này.',
            target_username='kingescaped',
            chat_id=TELEGRAM_ADMIN_ID,
            bot_token=TELEGRAM_BOT_TOKEN,
            group_id=TELEGRAM_GROUP_ID,
        )
else:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from hf_repo_sync import sync_base_to_private_repo
    sync_info = sync_base_to_private_repo(target_repo=TARGET_REPO, token=HF_TOKEN)
    print('Trạng thái repo Private:', sync_info)

In [ ]:
btc_metrics = None
gold_metrics = None

if READY_TO_TRAIN:
    from finetune_data import build_kronos_datasets, fetch_binance_btc_m15, fetch_gold_h1_candles, validate_recent_candles
    from finetune_trainer import FineTuneConfig, run_specialist_finetune

    try:
        btc_df = fetch_binance_btc_m15(days=365, symbol='BTCUSDT')
        btc_recent = validate_recent_candles(btc_df, val_days=30, min_bars=2000)
        print(f'Đã tải {len(btc_df):,} nến BTC M15 ({btc_recent:,} nến trong 30 ngày cuối)')
        btc_train, btc_val = build_kronos_datasets(btc_df, context_len=200, pred_len=12, stride=4, val_days=30)
        btc_cfg = FineTuneConfig(symbol='BTCUSD', branch='btc-m15', context_len=200, pred_len=12, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
        btc_metrics = run_specialist_finetune(btc_train, btc_val, btc_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
    except Exception as exc:
        print(f'[CẢNH BÁO BTC] Bỏ qua cập nhật nhánh btc-m15 do lỗi dữ liệu/huấn luyện: {exc}')
        btc_metrics = {'symbol': 'BTCUSD', 'branch': 'btc-m15', 'status': 'ANOMALY', 'anomaly': True, 'reason': str(exc), 'pushed': False}
    print('Kết quả BTC:', btc_metrics)

    try:
        gold_df = fetch_gold_h1_candles(period='730d', ticker='GC=F')
        gold_recent = validate_recent_candles(gold_df, val_days=30, min_bars=300)
        print(f'Đã tải {len(gold_df):,} nến Vàng H1 ({gold_recent:,} nến trong 30 ngày cuối)')
        gold_train, gold_val = build_kronos_datasets(gold_df, context_len=200, pred_len=6, stride=2, val_days=30)
        gold_cfg = FineTuneConfig(symbol='XAUUSD', branch='xau-h1', context_len=200, pred_len=6, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
        gold_metrics = run_specialist_finetune(gold_train, gold_val, gold_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
    except Exception as exc:
        print(f'[CẢNH BÁO XAU] Bỏ qua cập nhật nhánh xau-h1 do lỗi dữ liệu/huấn luyện: {exc}')
        gold_metrics = {'symbol': 'XAUUSD', 'branch': 'xau-h1', 'status': 'ANOMALY', 'anomaly': True, 'reason': str(exc), 'pushed': False}
    print('Kết quả Vàng:', gold_metrics)

In [ ]:
if READY_TO_TRAIN and btc_metrics and gold_metrics:
    import kronos_predictor
    from finetune_trainer import format_finetune_status_vn
    from telegram_notifier import send_telegram_direct_message

    kronos_predictor._GLOBAL_PREDICTOR = None
    pred_btc = kronos_predictor.load_kronos_base('BTCUSD', 'btc-m15')
    pred_xau = kronos_predictor.load_kronos_base('XAUUSD', 'xau-h1')
    print('Nguồn model BTC:', getattr(pred_btc, 'model_source', None))
    print('Nguồn model XAU:', getattr(pred_xau, 'model_source', None))

    if TELEGRAM_BOT_TOKEN:
        btc_line = format_finetune_status_vn(btc_metrics)
        xau_line = format_finetune_status_vn(gold_metrics)
        msg = (
            '🧠 <b>[AI LVQ Std BOT] - KẾT QUẢ FINE-TUNE ĐỊNH KỲ THÁNG (ĐỀ THI 30 NGÀY MỚI)</b>\n'
            '━━━━━━━━━━━━━━━━━━━━\n'
            f'🟠 <b>BTC (btc-m15):</b> {btc_line}\n'
            f'🟡 <b>XAU (xau-h1):</b> {xau_line}\n'
            '━━━━━━━━━━━━━━━━━━━━\n'
            '✅ <i>Chu kỳ quét tiếp theo sẽ tự động dùng bộ trọng số tốt nhất trên mỗi nhánh.</i>'
        )
        send_telegram_direct_message(
            msg, target_username='kingescaped', chat_id=TELEGRAM_ADMIN_ID, bot_token=TELEGRAM_BOT_TOKEN, group_id=TELEGRAM_GROUP_ID
        )